# 🎯 Phase 9: Milestone Exam Solutions

> **Enterprise SQL & Performance Engineering**
>
> This notebook contains comprehensive solutions for all five Phase Milestone Exam questions.
> SQL queries use Python's `sqlite3` module for Pyodide compatibility.
> PostgreSQL-specific features are explained in markdown cells.

---

## Question 1: Materialized Views & Caching Strategy

**Combines**: Views (Day 97), Performance Tuning (Day 106)

**Scenario**: Design a materialized view strategy for a high-traffic analytics dashboard that queries millions of rows but must respond in under 500ms.

### Theory: Standard View vs Materialized View

| Feature | Standard View | Materialized View |
|---------|--------------|-------------------|
| **Storage** | None (just a saved query) | Stores result set on disk |
| **Speed** | Same as running the query | Pre-computed (instant reads) |
| **Freshness** | Always current | Stale until refreshed |
| **Space** | Zero | Uses disk space |
| **Indexable** | No | Yes (index the mat view!) |
| **Best For** | Simple query reuse | Expensive aggregations |

### PostgreSQL Materialized View Syntax

```sql
-- Create materialized view
CREATE MATERIALIZED VIEW mv_daily_revenue AS
SELECT
    DATE(order_date) AS day,
    category,
    SUM(total) AS daily_revenue,
    COUNT(*) AS order_count,
    AVG(total) AS avg_order_value
FROM orders
GROUP BY DATE(order_date), category;

-- Index the materialized view for fast lookups
CREATE INDEX idx_mv_daily_day ON mv_daily_revenue(day);
CREATE INDEX idx_mv_daily_cat ON mv_daily_revenue(category);

-- Refresh (blocking — locks reads during refresh)
REFRESH MATERIALIZED VIEW mv_daily_revenue;

-- Refresh CONCURRENTLY (non-blocking — requires unique index)
CREATE UNIQUE INDEX idx_mv_daily_unique ON mv_daily_revenue(day, category);
REFRESH MATERIALIZED VIEW CONCURRENTLY mv_daily_revenue;
```

### Refresh Strategy Decision Tree

| Requirement | Strategy | Implementation |
|-------------|----------|---------------|
| Dashboard (updated daily) | Scheduled refresh | `pg_cron` job at 2 AM |
| Near real-time | Trigger-based invalidation | Refresh on source table change |
| Always fresh | Standard view + good indexes | No materialization needed |
| Mixed (some fresh, some cached) | Hybrid | Critical KPIs → live; historical → mat view |

In [ ]:
import sqlite3
import random
import time

conn = sqlite3.connect(":memory:")
cur = conn.cursor()

# Create a large-ish orders table
cur.execute("""
    CREATE TABLE orders (
        id INTEGER PRIMARY KEY,
        order_date TEXT,
        category TEXT,
        total REAL,
        customer_id INTEGER
    )
""")

random.seed(42)
categories = ['Electronics', 'Books', 'Clothing', 'Home', 'Food', 'Sports']
for i in range(1, 5001):
    cur.execute("INSERT INTO orders VALUES (?,?,?,?,?)",
        (i, f"2024-{random.randint(1,12):02d}-{random.randint(1,28):02d}",
         random.choice(categories),
         round(random.uniform(10, 500), 2),
         random.randint(1, 500)))
conn.commit()

# Simulate materialized view (SQLite doesn't support MATERIALIZED VIEW)
# We'll create a regular table that acts as the cached result

def refresh_materialized_view(cursor, conn):
    """Simulate REFRESH MATERIALIZED VIEW."""
    cursor.execute("DROP TABLE IF EXISTS mv_daily_revenue")
    cursor.execute("""
        CREATE TABLE mv_daily_revenue AS
        SELECT
            SUBSTR(order_date, 1, 7) AS month,
            category,
            SUM(total) AS monthly_revenue,
            COUNT(*) AS order_count,
            ROUND(AVG(total), 2) AS avg_order_value
        FROM orders
        GROUP BY SUBSTR(order_date, 1, 7), category
    """)
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_mv_month ON mv_daily_revenue(month)")
    conn.commit()


# Benchmark: Direct query vs Materialized View
print("=" * 55)
print("MATERIALIZED VIEW PERFORMANCE COMPARISON")
print("=" * 55)

# Direct query
t0 = time.time()
for _ in range(100):
    cur.execute("""
        SELECT SUBSTR(order_date, 1, 7), category,
               SUM(total), COUNT(*), AVG(total)
        FROM orders
        GROUP BY SUBSTR(order_date, 1, 7), category
    """).fetchall()
direct_time = (time.time() - t0) / 100 * 1000

# Materialized view
refresh_materialized_view(cur, conn)

t0 = time.time()
for _ in range(100):
    cur.execute("SELECT * FROM mv_daily_revenue").fetchall()
mv_time = (time.time() - t0) / 100 * 1000

speedup = direct_time / mv_time if mv_time > 0 else float('inf')

print(f"\n  Direct query (5K rows): {direct_time:.2f} ms avg")
print(f"  Materialized view:      {mv_time:.2f} ms avg")
print(f"  Speedup:                {speedup:.1f}x faster")
print(f"\n💡 With millions of rows, the speedup is 100-1000x.")

# Show the materialized view contents
print(f"\n{'Month':>8s} {'Category':>12s} {'Revenue':>12s} {'Orders':>8s} {'AOV':>8s}")
print("-" * 52)
rows = cur.execute("SELECT * FROM mv_daily_revenue ORDER BY month, category LIMIT 12").fetchall()
for m, cat, rev, cnt, aov in rows:
    print(f"{m:>8s} {cat:>12s} ${rev:>10,.0f} {cnt:>8d} ${aov:>6,.2f}")

---

## Question 2: Advanced Indexing — Composite & Partial

**Combines**: Advanced Indexing (Day 98), Query Planning (Day 106)

**Scenario**: Design an optimal indexing strategy for these common query patterns:

```sql
-- Pattern 1: Active orders for a customer in date range
SELECT * FROM orders
WHERE customer_id = ? AND status = 'active'
  AND order_date BETWEEN ? AND ?;

-- Pattern 2: Top spenders (aggregate with filter)
SELECT customer_id, SUM(total)
FROM orders WHERE status != 'cancelled'
GROUP BY customer_id HAVING SUM(total) > 1000;

-- Pattern 3: Full-text search on product names
SELECT * FROM products WHERE name ILIKE '%wireless%';
```

### Index Selection Rules

1. **Equality columns first**, then range columns in composite index
2. **Most selective column first** (the one that filters out the most rows)
3. **Partial indexes** for queries that always filter on a constant value
4. **Covering indexes** (INCLUDE) to avoid table lookups

### PostgreSQL-Specific Index Types

```sql
-- Composite index (equality + range best practice)
CREATE INDEX idx_orders_lookup
    ON orders(customer_id, status, order_date);
-- Reads: customer_id=? AND status=? (equality prefix) then range on date

-- Partial index (only index non-cancelled orders)
CREATE INDEX idx_orders_active
    ON orders(customer_id, order_date)
    WHERE status != 'cancelled';
-- Smaller index, faster scans, only covers relevant queries

-- GIN index for full-text search
CREATE INDEX idx_products_search
    ON products USING GIN(to_tsvector('english', name));
-- Enables: WHERE to_tsvector('english', name) @@ to_tsquery('wireless')

-- Covering index (avoids heap lookup)
CREATE INDEX idx_orders_covering
    ON orders(customer_id)
    INCLUDE (total, status, order_date);
-- Index-only scan: all needed columns are IN the index
```

In [ ]:
# Demo: Composite index effectiveness
print("=" * 55)
print("COMPOSITE INDEX DEMO")
print("=" * 55)

# Add status column to orders
statuses = ['active', 'shipped', 'delivered', 'cancelled']
cur.execute("ALTER TABLE orders ADD COLUMN status TEXT DEFAULT 'delivered'")
for i in range(1, 5001):
    cur.execute("UPDATE orders SET status = ? WHERE id = ?",
               (random.choice(statuses), i))
conn.commit()

test_query = """SELECT id, order_date, total FROM orders
WHERE customer_id = 42 AND status = 'active'
AND order_date BETWEEN '2024-03-01' AND '2024-06-30'"""

# No index
plan = cur.execute(f"EXPLAIN QUERY PLAN {test_query}").fetchall()
print(f"\n  No index:     {plan[0][-1]}")

# Single column index
cur.execute("CREATE INDEX idx_cust ON orders(customer_id)")
plan = cur.execute(f"EXPLAIN QUERY PLAN {test_query}").fetchall()
print(f"  Single index: {plan[0][-1]}")

# Composite index (optimal order: equality, equality, range)
cur.execute("CREATE INDEX idx_composite ON orders(customer_id, status, order_date)")
plan = cur.execute(f"EXPLAIN QUERY PLAN {test_query}").fetchall()
print(f"  Composite:    {plan[0][-1]}")

print("\n💡 The composite index serves all three WHERE conditions efficiently:")
print("   customer_id = 42 (equality) → status = 'active' (equality) → date range")

---

## Question 3: JSON Data Handling in SQL

**Combines**: JSON Handling (Day 101), Semi-Structured Data (Day 102)

**Scenario**: Work with JSON data stored in a SQL database. Extract, query, and aggregate JSON fields.

### Theory: When to Use JSON in SQL

| Use JSON When | Don't Use JSON When |
|---------------|--------------------|
| Schema varies per row (user preferences) | Data has consistent structure |
| Nested data (order items, tags) | You need to JOIN on the field |
| API response storage | You need to index for fast lookup |
| Rapid prototyping | Performance is critical |

In [ ]:
import json

# Create a table with JSON data
cur.executescript("""
    CREATE TABLE events (
        id INTEGER PRIMARY KEY,
        event_type TEXT,
        timestamp TEXT,
        payload TEXT  -- JSON stored as TEXT in SQLite
    );
""")

# Insert events with JSON payloads
events = [
    (1, 'page_view', '2024-03-15 10:30:00',
     json.dumps({"url": "/products/widget", "user_id": 42, "device": "mobile", "duration_sec": 45})),
    (2, 'purchase', '2024-03-15 10:35:00',
     json.dumps({"order_id": 1001, "items": [{"name": "Widget", "qty": 2, "price": 29.99}], "total": 59.98, "payment": "credit_card"})),
    (3, 'page_view', '2024-03-15 11:00:00',
     json.dumps({"url": "/products/gadget", "user_id": 43, "device": "desktop", "duration_sec": 120})),
    (4, 'purchase', '2024-03-15 11:15:00',
     json.dumps({"order_id": 1002, "items": [{"name": "Gadget", "qty": 1, "price": 49.99}, {"name": "Case", "qty": 1, "price": 9.99}], "total": 59.98, "payment": "paypal"})),
    (5, 'signup', '2024-03-15 12:00:00',
     json.dumps({"user_id": 44, "source": "google_ads", "plan": "free", "referral_code": "FRIEND10"})),
    (6, 'page_view', '2024-03-15 14:00:00',
     json.dumps({"url": "/products/widget", "user_id": 42, "device": "mobile", "duration_sec": 30})),
    (7, 'purchase', '2024-03-15 14:30:00',
     json.dumps({"order_id": 1003, "items": [{"name": "Widget Pro", "qty": 1, "price": 99.99}], "total": 99.99, "payment": "credit_card"})),
]

cur.executemany("INSERT INTO events VALUES (?,?,?,?)", events)
conn.commit()
print("✅ Events loaded with JSON payloads")

In [ ]:
# Query JSON data using SQLite's json_extract
print("=" * 55)
print("JSON QUERIES")
print("=" * 55)

# 1. Extract JSON fields
print("\n📋 Page Views (extracting JSON fields):")
rows = cur.execute("""
    SELECT
        timestamp,
        json_extract(payload, '$.url') AS url,
        json_extract(payload, '$.user_id') AS user_id,
        json_extract(payload, '$.device') AS device,
        json_extract(payload, '$.duration_sec') AS duration
    FROM events
    WHERE event_type = 'page_view'
    ORDER BY timestamp;
""").fetchall()
print(f"  {'Time':>20s} {'URL':>20s} {'User':>6s} {'Device':>8s} {'Dur':>5s}")
for ts, url, uid, dev, dur in rows:
    print(f"  {ts:>20s} {url:>20s} {uid:>6d} {dev:>8s} {dur:>4d}s")

# 2. Aggregate JSON data
print("\n💰 Revenue by Payment Method:")
rows = cur.execute("""
    SELECT
        json_extract(payload, '$.payment') AS payment,
        COUNT(*) AS orders,
        SUM(json_extract(payload, '$.total')) AS revenue
    FROM events
    WHERE event_type = 'purchase'
    GROUP BY payment;
""").fetchall()
for pay, cnt, rev in rows:
    print(f"  {pay:>15s}: {cnt} orders, ${rev:,.2f} revenue")

# 3. Query nested JSON (items array)
print("\n📦 Order Items (nested JSON array):")
rows = cur.execute("""
    SELECT
        json_extract(payload, '$.order_id') AS order_id,
        json_extract(payload, '$.total') AS total,
        json_extract(payload, '$.items') AS items_json
    FROM events
    WHERE event_type = 'purchase'
""").fetchall()

for oid, total, items_json in rows:
    items = json.loads(items_json)
    print(f"  Order #{oid} (${total:.2f}):")
    for item in items:
        print(f"    - {item['name']} × {item['qty']} @ ${item['price']:.2f}")

### PostgreSQL JSON Operators

```sql
-- PostgreSQL uses -> and ->> operators (more readable)
-- -> returns JSON, ->> returns text

SELECT
    payload->>'url' AS url,                    -- Text extraction
    (payload->>'duration_sec')::INT AS dur,     -- Cast to integer
    payload->'items'->0->>'name' AS first_item, -- Nested array access
    jsonb_array_length(payload->'items') AS item_count
FROM events;

-- Cross-join with JSON array (unnest items)
SELECT
    e.id,
    item->>'name' AS product,
    (item->>'qty')::INT AS quantity,
    (item->>'price')::NUMERIC AS price
FROM events e,
     jsonb_array_elements(e.payload->'items') AS item
WHERE e.event_type = 'purchase';

-- GIN index for JSON containment queries
CREATE INDEX idx_events_payload ON events USING GIN(payload jsonb_path_ops);
-- Enables: WHERE payload @> '{"payment": "credit_card"}'
```

---

## Question 4: Database Security & Row-Level Security

**Combines**: Security (Day 103), Access Control (Day 104)

**Scenario**: Design a multi-tenant security model where:
- Each customer can only see their own data
- Admins can see all data
- API keys have different permission levels

### Theory: Defense in Depth

```
┌─────────────────────────────────┐
│     Application Firewall        │  Layer 1: Network
├─────────────────────────────────┤
│     Authentication (SSL/TLS)    │  Layer 2: Connection
├─────────────────────────────────┤
│     Role-Based Access (RBAC)    │  Layer 3: Database
├─────────────────────────────────┤
│     Row-Level Security (RLS)    │  Layer 4: Data
├─────────────────────────────────┤
│     Column Encryption           │  Layer 5: Field
└─────────────────────────────────┘
```

### PostgreSQL Row-Level Security

```sql
-- Enable RLS on the table
ALTER TABLE customer_data ENABLE ROW LEVEL SECURITY;

-- Policy: Users can only see their own tenant's data
CREATE POLICY tenant_isolation ON customer_data
    USING (tenant_id = current_setting('app.tenant_id')::INT);

-- Policy: Admins can see everything
CREATE POLICY admin_access ON customer_data
    FOR ALL
    TO admin_role
    USING (true);

-- Set tenant context per connection:
SET app.tenant_id = '42';  -- App middleware sets this
SELECT * FROM customer_data;  -- Only sees tenant 42's data!
```

In [ ]:
# Simulated Row-Level Security in Python/SQLite

class SecureDatabase:
    """
    Simulates PostgreSQL RLS using Python-side filtering.

    In production, this would be handled by PostgreSQL's
    built-in RLS policies at the database level.
    """

    def __init__(self):
        self.conn = sqlite3.connect(":memory:")
        self.cur = self.conn.cursor()
        self._setup()
        self._current_tenant = None
        self._is_admin = False

    def _setup(self):
        self.cur.executescript("""
            CREATE TABLE tenants (
                id INTEGER PRIMARY KEY, name TEXT
            );
            CREATE TABLE customer_data (
                id INTEGER PRIMARY KEY,
                tenant_id INTEGER REFERENCES tenants(id),
                customer_name TEXT,
                revenue REAL,
                email TEXT
            );
            INSERT INTO tenants VALUES (1, 'Acme Corp'), (2, 'Globex Inc'), (3, 'Initech');
            INSERT INTO customer_data VALUES
                (1, 1, 'Alice', 50000, 'alice@acme.com'),
                (2, 1, 'Bob', 75000, 'bob@acme.com'),
                (3, 2, 'Carol', 120000, 'carol@globex.com'),
                (4, 2, 'David', 90000, 'david@globex.com'),
                (5, 3, 'Elena', 65000, 'elena@initech.com'),
                (6, 3, 'Frank', 45000, 'frank@initech.com');
        """)
        self.conn.commit()

    def set_context(self, tenant_id=None, is_admin=False):
        """Set the current security context (like SET app.tenant_id)."""
        self._current_tenant = tenant_id
        self._is_admin = is_admin

    def query(self, base_query):
        """Execute query with RLS policy applied."""
        if self._is_admin:
            # Admin: no filter
            return self.cur.execute(base_query).fetchall()
        elif self._current_tenant:
            # Tenant: filter by tenant_id
            filtered = f"{base_query} WHERE tenant_id = {self._current_tenant}"
            # Handle existing WHERE clause
            if 'WHERE' in base_query.upper():
                filtered = f"{base_query} AND tenant_id = {self._current_tenant}"
            return self.cur.execute(filtered).fetchall()
        else:
            raise PermissionError("No security context set")


# Demo
print("=" * 55)
print("ROW-LEVEL SECURITY DEMO")
print("=" * 55)

db = SecureDatabase()
base = "SELECT id, customer_name, revenue, email FROM customer_data"

# Admin sees everything
db.set_context(is_admin=True)
print("\n👑 Admin View (all data):")
for row in db.query(base):
    print(f"  #{row[0]} {row[1]:>8s}  ${row[2]:>10,.0f}  {row[3]}")

# Tenant 1 (Acme) sees only their data
db.set_context(tenant_id=1)
print("\n🏢 Acme Corp View (tenant_id=1):")
for row in db.query(base):
    print(f"  #{row[0]} {row[1]:>8s}  ${row[2]:>10,.0f}  {row[3]}")

# Tenant 2 (Globex) sees only their data
db.set_context(tenant_id=2)
print("\n🏢 Globex Inc View (tenant_id=2):")
for row in db.query(base):
    print(f"  #{row[0]} {row[1]:>8s}  ${row[2]:>10,.0f}  {row[3]}")

print("\n🔒 Each tenant is completely isolated from the others.")
print("   In PostgreSQL, this is enforced at the database engine level (RLS).")

---

## Question 5: Performance Engineering — Distributed Queries

**Combines**: Performance Tuning (Day 106), Scaling (Day 107-108)

**Scenario**: Design a partitioning and scaling strategy for a time-series table with 1 billion rows.

### Theory: Partitioning Strategies

| Strategy | When to Use | Example |
|----------|------------|--------|
| **Range** | Time-series, sequential data | Partition by month/year |
| **Hash** | Distribute evenly | Partition by customer_id |
| **List** | Known categories | Partition by region |

### PostgreSQL Table Partitioning

```sql
-- Declarative partitioning (PostgreSQL 10+)
CREATE TABLE events (
    id BIGINT GENERATED ALWAYS AS IDENTITY,
    event_date DATE NOT NULL,
    user_id INT,
    event_type TEXT,
    payload JSONB
) PARTITION BY RANGE (event_date);

-- Monthly partitions
CREATE TABLE events_2024_01 PARTITION OF events
    FOR VALUES FROM ('2024-01-01') TO ('2024-02-01');
CREATE TABLE events_2024_02 PARTITION OF events
    FOR VALUES FROM ('2024-02-01') TO ('2024-03-01');
-- ... etc

-- Automatic partition creation (pg_partman extension)
SELECT partman.create_parent(
    p_parent_table := 'public.events',
    p_control := 'event_date',
    p_type := 'native',
    p_interval := '1 month'
);
```

### Scaling Decision Matrix

| Data Size | Strategy | Tools |
|-----------|----------|-------|
| < 10M rows | Single PostgreSQL + indexes | Standard |
| 10M - 1B rows | Partitioning + read replicas | pg_partman, Patroni |
| > 1B rows | Distributed database | Citus, CockroachDB, YugabyteDB |
| > 10B rows | Data warehouse | BigQuery, Snowflake, Redshift |

In [ ]:
# Simulated partition pruning demo
print("=" * 55)
print("PARTITION PRUNING DEMO")
print("=" * 55)

# Simulate partitioned table with separate tables per month
months_data = {}
total_rows = 0

random.seed(42)
for m in range(1, 13):
    month = f"2024-{m:02d}"
    n_rows = random.randint(800, 1200)
    months_data[month] = n_rows
    total_rows += n_rows

    cur.execute(f"DROP TABLE IF EXISTS events_{m:02d}")
    cur.execute(f"""
        CREATE TABLE events_{m:02d} (
            id INTEGER, event_date TEXT, user_id INTEGER, event_type TEXT
        )
    """)

    for i in range(n_rows):
        cur.execute(f"INSERT INTO events_{m:02d} VALUES (?,?,?,?)",
            (i, f"{month}-{random.randint(1,28):02d}",
             random.randint(1, 1000),
             random.choice(['click', 'view', 'purchase'])))

conn.commit()

print(f"\nTotal rows across 12 partitions: {total_rows:,}")
print(f"\n{'Partition':>15s} {'Rows':>8s}")
print("-" * 25)
for month, n in months_data.items():
    print(f"{month:>15s} {n:>8,d}")

# Simulate a query that only needs one partition
target_month = 3
print(f"\n📊 Query: Events in March 2024")
print(f"   Without partitioning: Scan {total_rows:,} rows")
print(f"   With partitioning:    Scan {months_data['2024-03']:,} rows (partition events_03 only)")
print(f"   Savings: {100 * (1 - months_data['2024-03'] / total_rows):.1f}% fewer rows scanned")

# Run the query on the single partition
t0 = time.time()
result = cur.execute(f"""
    SELECT event_type, COUNT(*) AS count
    FROM events_{target_month:02d}
    GROUP BY event_type
    ORDER BY count DESC
""").fetchall()
elapsed = (time.time() - t0) * 1000

print(f"\n   Results (partition events_{target_month:02d}, {elapsed:.2f}ms):")
for event_type, count in result:
    print(f"     {event_type:>10s}: {count:,d}")

---

## 🎓 Summary

This notebook demonstrated solutions to all five Phase 9 Milestone Exam questions:

1. **Materialized Views**: Creation, refresh strategies (blocking vs concurrent), performance benchmarks
2. **Advanced Indexing**: Composite, partial, GIN, covering indexes with EXPLAIN verification
3. **JSON Handling**: json_extract queries, nested array access, PostgreSQL JSONB operators
4. **Database Security**: Row-Level Security simulation, multi-tenant isolation, defense in depth
5. **Performance Engineering**: Partitioning strategies, partition pruning, scaling decision matrix

Phase 9 master skill: **Designing systems that scale**. The difference between a database that handles 1K users and 1M users is architecture, not just code.